# Multimodal Meme Understanding: Image, Text, or Both?
### Deep Learning Final Project
This notebook reproduces the complete experimental pipeline:
1. **Data:** Download Memotion 7k dataset (SemEval-2020 Task 8: 3-class Sentiment Analysis)
2. **Features:** Extract and cache frozen CLIP ViT-L/14 embeddings (768-d)
3. **Linear Probes & Fusion Heads:** Image-only, Text-only, Concat, Product, and Gated Fusion (5 seeds)
4. **Fine-Tuned Models:** ViT-B/16 (Image), BERT-base (Text), Partially-unfrozen CLIP (Multimodal)
5. **Evaluation:** Official Macro F1-score & Accuracy on Validation and Test sets


## 1. Environment Setup & Google Drive Mount


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil

REPO_URL = 'https://github.com/arcrek/dl2026.git'
ROOT = '/content/drive/MyDrive/dl2026_memotion'
os.environ['DATA_DIR'] = '/content/data/memotion'
os.environ['FEATURE_DIR'] = f'{ROOT}/features'
os.environ['RESULTS_DIR'] = f'{ROOT}/results'

for d in ('features', 'results', 'configs'):
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

if os.path.exists('/content/repo'):
    !git -C /content/repo pull
else:
    !git clone $REPO_URL /content/repo
%cd /content/repo

!pip install -q transformers scikit-learn pyarrow
os.makedirs('configs', exist_ok=True)
if os.path.exists(f'{ROOT}/configs/heads.json'):
    shutil.copy(f'{ROOT}/configs/heads.json', 'configs/heads.json')


## 2. Dataset Download & Verification
Restores dataset from Drive tar if available, or downloads Memotion 7k directly via `scripts/download_data.sh`.


In [ ]:
TAR = f'{ROOT}/memotion.tar'
if os.path.exists(TAR) and not os.path.exists('/content/data/memotion/train.jsonl'):
    !mkdir -p /content/data && tar -xf "$TAR" -C /content/data

!bash scripts/download_data.sh
!python src/data.py --images

if not os.path.exists(TAR):
    !tar -cf "$TAR" -C /content/data memotion


## 3. Extract & Cache CLIP ViT-L/14 Features
Extracts 768-d visual and textual embeddings for all splits and saves them directly to Google Drive.


In [ ]:
!python src/features.py


## 4. Smoke Tests


In [ ]:
!python tests/test_data.py
!python tests/test_heads.py


## 5. Linear Probes and Fusion Heads (5 seeds: 0, 1, 2, 3, 4)
Trains image-only, text-only, concat linear, concat MLP, product MLP, and gated fusion heads.


In [ ]:
# Auto-tunes hyperparameters on group-aware hold-out if needed, then evaluates 5 seeds
!python src/heads.py --run
if os.path.exists('configs/heads.json'):
    shutil.copy('configs/heads.json', f'{ROOT}/configs/heads.json')


## 6. Fine-Tuned Models (3 seeds: 0, 1, 2)
Evaluates full backbone representations for comparison against frozen CLIP probes.


In [ ]:
# Image-only Fine-Tuning: ViT-B/16
!python src/finetune.py --mode vit --seeds 0 1 2


In [ ]:
# Text-only Fine-Tuning: BERT-base
!python src/finetune.py --mode bert --seeds 0 1 2


In [ ]:
# Multimodal Fine-Tuning: Partially unfrozen CLIP ViT-L/14 with Gated Fusion
!python src/finetune.py --mode clip --seeds 0 1 2


## 7. Results Aggregation & Summary Table


In [ ]:
import json, glob, pandas as pd

rows = []
all_results = sorted(glob.glob(f'{ROOT}/results/heads/*.json') + glob.glob(f'{ROOT}/results/finetune/*.json'))
for f in all_results:
    r = json.load(open(f))
    row = {'model': r['model'], 'seed': r['seed']}
    for s in ['holdout', 'validation', 'test']:
        for m in ['macro_f1', 'acc']:
            row[f'{s}_{m}'] = r[s][m]
    rows.append(row)

df = pd.DataFrame(rows)
agg = df.drop(columns='seed').groupby('model').agg(['mean', 'std']).round(3)
agg['n_seeds'] = df.groupby('model').size()
summary_csv = f'{ROOT}/results/phase2_summary.csv'
agg.to_csv(summary_csv)
print("Saved summary table to:", summary_csv)
agg
